In [78]:
import operator
import random
from typing import TypedDict, Annotated
from pprint import pprint
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

In [79]:
class ProcessState(TypedDict):
    title: str
    sub_process: list[str]
    comments: Annotated[list[str], operator.add]
    results: str

class SubProcessReview(TypedDict):
    sub_process_data: str

In [80]:
def generate_topic(state: ProcessState):
    topic = state["title"]
    sub_process = [f"{topic}- {1}", f"{topic}- {2}", f"{topic}- {3}"]
    print(f"[Sub-Process: {sub_process}]")
    return {
        "sub_process": sub_process
    }


def review_process(state: SubProcessReview):
    data = state["sub_process_data"]
    print(f"[Review-Process: {data}]")
    comm = data + " - " + random.choice(["Good", "Pass", "Excellent"])
    return {
        "comments": [comm]
    }


def process_status(state: ProcessState):
    print(f"[Status-Process]")
    resp = "*" * 25
    resp += f"\n\n {state["title"]}\n"
    resp += "*" * 25
    resp += f"\n\n Comments: {state["comments"]}"
    return {
        "results": resp
    }


def split_and_execute_process(state: ProcessState):
    return [
        Send("review_process", {"sub_process_data": sp}) for sp in state["sub_process"]
    ]

In [81]:
workflow_builder = StateGraph(ProcessState)
workflow_builder.add_node("generate_topic", generate_topic)
workflow_builder.add_node("review_process", review_process)
workflow_builder.add_node("process_status", process_status)

workflow_builder.add_edge(START, "generate_topic")
workflow_builder.add_conditional_edges("generate_topic", split_and_execute_process, ["review_process"])
workflow_builder.add_edge("review_process", "process_status")
workflow_builder.add_edge("process_status", END)
graph = workflow_builder.compile()

In [82]:
response = graph.invoke(input={
    "title":"Build process"
})
pprint(response)
print(response["results"])

[Sub-Process: ['Build process- 1', 'Build process- 2', 'Build process- 3']]
[Review-Process: Build process- 1]
[Review-Process: Build process- 3]
[Review-Process: Build process- 2]
[Status-Process]
{'comments': ['Build process- 1 - Excellent',
              'Build process- 2 - Good',
              'Build process- 3 - Good'],
 'results': '*************************\n'
            '\n'
            ' Build process\n'
            '*************************\n'
            '\n'
            " Comments: ['Build process- 1 - Excellent', 'Build process- 2 - "
            "Good', 'Build process- 3 - Good']",
 'sub_process': ['Build process- 1', 'Build process- 2', 'Build process- 3'],
 'title': 'Build process'}
*************************

 Build process
*************************

 Comments: ['Build process- 1 - Excellent', 'Build process- 2 - Good', 'Build process- 3 - Good']
